In [1]:
import pandas as pd
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)


def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
        return df

In [2]:
query = """
SELECT
    DATE_TRUNC('month', cd.book_date)       AS book_month,
    cd.pb_current_state_code                AS state,
    COUNT(*)                                AS loan_count
FROM edwnpi.los_deal_current_fact AS cd
LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
       ON dru.dealer_number = cd.dealer_number
WHERE cd.data_source_name != 'SPARTAN'
  AND cd.book_date IS NOT NULL
  AND cd.book_date >= '2026-01-01'
  AND dru.riskdealergroup = 'AN'
GROUP BY 1, 2
ORDER BY 1, 2
"""

df = run_sql(query)
df['book_month'] = pd.to_datetime(df['book_month'])
print(f"Rows returned: {len(df)}")
df.head(10)

Rows returned: 183


,book_month,state,loan_count
0,2026-01-01,AL,18
1,2026-01-01,AZ,57
2,2026-01-01,CA,47
3,2026-01-01,CO,33
4,2026-01-01,CT,1
5,2026-01-01,FL,173
6,2026-01-01,Fl,1
7,2026-01-01,GA,18
8,2026-01-01,IA,1
9,2026-01-01,ID,2


In [3]:
df['month_label'] = df['book_month'].dt.strftime('%Y-%m')

# Pivot: loan counts by month and state
counts_pivot = df.pivot_table(
    index='state', columns='month_label', values='loan_count',
    aggfunc='sum', fill_value=0
)
counts_pivot['Total'] = counts_pivot.sum(axis=1)
counts_pivot = counts_pivot.sort_values('Total', ascending=False)

# Percentage distribution within each month (column-normalized)
month_cols = [c for c in counts_pivot.columns if c != 'Total']
pct_pivot = counts_pivot[month_cols].apply(lambda col: col / col.sum() * 100)
pct_pivot['Total'] = counts_pivot['Total'] / counts_pivot['Total'].sum() * 100

In [4]:
print("=" * 80)
print("AN LOB - State Distribution by Month (Loan Counts)")
print("=" * 80)
print(counts_pivot.to_string())

print("\n")
print("=" * 80)
print("AN LOB - State Distribution by Month (Percentages)")
print("=" * 80)
print(pct_pivot.round(2).to_string())

print("\n")
print("=" * 80)
print("Monthly Totals")
print("=" * 80)
monthly_totals = counts_pivot[month_cols].sum()
print(monthly_totals.to_string())

AN LOB - State Distribution by Month (Loan Counts)
month_label  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07  Total
state                                                                            
TX               132      199      333      304      313      318      207   1806
FL               173      159      285      220      228      229      183   1477
AZ                57       67       88       69       66       61       49    457
CO                33       37       52       45       40       81       91    379
NV                52       39       69       54       41       51       36    342
CA                47       44       56       53       38       40       40    318
GA                18       31       45       36       43       40       29    242
AL                18       16       28       16       27        5       12    122
WA                10       15       23        8        7       22       14     99
SC                 3       12       24       18

In [5]:
apr_query = """
SELECT
    DATE_TRUNC('month', cd.book_date)       AS book_month,
    cd.pb_current_state_code                AS state,
    COUNT(*)                                AS loan_count,
    SUM(cd.con_amount_financed_back)        AS total_amt_financed,
    SUM(cd.con_amount_financed_back * cd.con_apr) AS apr_x_amt
FROM edwnpi.los_deal_current_fact AS cd
LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
       ON dru.dealer_number = cd.dealer_number
WHERE cd.data_source_name != 'SPARTAN'
  AND cd.book_date IS NOT NULL
  AND cd.book_date >= '2026-01-01'
  AND dru.riskdealergroup = 'AN'
  AND cd.con_apr IS NOT NULL
  AND cd.con_amount_financed_back > 0
GROUP BY 1, 2
ORDER BY 1, 2
"""

apr_df = run_sql(apr_query)
apr_df['book_month'] = pd.to_datetime(apr_df['book_month'])
apr_df['state'] = apr_df['state'].str.upper().str.strip()
apr_df['weighted_apr'] = apr_df['apr_x_amt'] / apr_df['total_amt_financed']
apr_df['month_label'] = apr_df['book_month'].dt.strftime('%Y-%m')
print(f"Rows returned: {len(apr_df)}")
apr_df.head(10)

Rows returned: 183


,book_month,state,loan_count,total_amt_financed,apr_x_amt,weighted_apr,month_label
0,2026-01-01,AL,18,323706.39,90587.195163,0.279844,2026-01
1,2026-01-01,AZ,57,1181930.93,329185.900207,0.278515,2026-01
2,2026-01-01,CA,47,833353.46,233255.633454,0.279900,2026-01
3,2026-01-01,CO,33,746497.86,160964.819232,0.215627,2026-01
4,2026-01-01,CT,1,16795.04,4700.931696,0.279900,2026-01
5,2026-01-01,FL,173,4418849.32,974379.164529,0.220505,2026-01
6,2026-01-01,FL,1,23803.18,5653.255250,0.237500,2026-01
7,2026-01-01,GA,18,429767.51,119919.118413,0.279033,2026-01
8,2026-01-01,IA,1,41372.63,8522.761780,0.206000,2026-01
9,2026-01-01,ID,2,55753.17,15605.312283,0.279900,2026-01


In [6]:
# Overall weighted APR by month
overall = apr_df.groupby('month_label').agg(
    total_amt=('total_amt_financed', 'sum'),
    apr_x_amt=('apr_x_amt', 'sum'),
    loans=('loan_count', 'sum')
).reset_index()
overall['weighted_apr'] = overall['apr_x_amt'] / overall['total_amt']

print("=" * 60)
print("AN LOB - Overall Weighted APR by Month")
print("=" * 60)
print(overall[['month_label', 'weighted_apr', 'loans']].to_string(index=False))
print(f"\nAPR change (first to last): {(overall['weighted_apr'].iloc[-1] - overall['weighted_apr'].iloc[0]):.5f}")
print(f"  = {(overall['weighted_apr'].iloc[-1] - overall['weighted_apr'].iloc[0]) * 100:.2f} bps")

AN LOB - Overall Weighted APR by Month
month_label  weighted_apr  loans
    2026-01      0.240238    583
    2026-02      0.244198    684
    2026-03      0.241473   1105
    2026-04      0.236608    886
    2026-05      0.228775    875
    2026-06      0.228094    914
    2026-07      0.225625    710

APR change (first to last): -0.01461
  = -1.46 bps


In [10]:
# Shift-share decomposition: first full month vs last full month
# Decomposes the overall APR change into:
#   mix_effect:  state's volume share changed (holding its APR constant at period avg)
#   rate_effect: state's APR changed (holding its volume share constant at period avg)

# Re-aggregate by normalized state and month to collapse case duplicates (e.g. "FL"/"Fl")
state_month = apr_df.groupby(['month_label', 'state']).agg(
    total_amt_financed=('total_amt_financed', 'sum'),
    apr_x_amt=('apr_x_amt', 'sum'),
    loan_count=('loan_count', 'sum')
).reset_index()
state_month['weighted_apr'] = state_month['apr_x_amt'] / state_month['total_amt_financed']

months_sorted = sorted(state_month['month_label'].unique())
first_month = months_sorted[0]
last_month = months_sorted[-1]

first = state_month[state_month['month_label'] == first_month].set_index('state')
last = state_month[state_month['month_label'] == last_month].set_index('state')

all_states = sorted(set(first.index) | set(last.index))

decomp_rows = []
for st in all_states:
    af_0 = first.loc[st, 'total_amt_financed'] if st in first.index else 0
    af_1 = last.loc[st, 'total_amt_financed'] if st in last.index else 0
    apr_0 = first.loc[st, 'weighted_apr'] if st in first.index else 0
    apr_1 = last.loc[st, 'weighted_apr'] if st in last.index else 0

    decomp_rows.append({
        'state': st,
        'amt_fin_first': af_0,
        'amt_fin_last': af_1,
        'apr_first': apr_0,
        'apr_last': apr_1,
    })

decomp = pd.DataFrame(decomp_rows).set_index('state')

total_af_first = decomp['amt_fin_first'].sum()
total_af_last = decomp['amt_fin_last'].sum()

decomp['share_first'] = decomp['amt_fin_first'] / total_af_first
decomp['share_last'] = decomp['amt_fin_last'] / total_af_last
decomp['avg_share'] = (decomp['share_first'] + decomp['share_last']) / 2
decomp['avg_apr'] = (decomp['apr_first'] + decomp['apr_last']) / 2

# Shift-share: APR_change = sum_s [ delta_share_s * avg_apr_s ] + sum_s [ avg_share_s * delta_apr_s ]
decomp['delta_share'] = decomp['share_last'] - decomp['share_first']
decomp['delta_apr'] = decomp['apr_last'] - decomp['apr_first']

decomp['mix_effect'] = decomp['delta_share'] * decomp['avg_apr']
decomp['rate_effect'] = decomp['avg_share'] * decomp['delta_apr']
decomp['total_effect'] = decomp['mix_effect'] + decomp['rate_effect']

decomp = decomp.sort_values('total_effect')

print("=" * 80)
print(f"Shift-Share Decomposition: {first_month} vs {last_month}")
print("=" * 80)
print(f"{'State':<6} {'Share_1':>8} {'Share_T':>8} {'APR_1':>8} {'APR_T':>8} {'Mix_Eff':>9} {'Rate_Eff':>9} {'Total':>9}")
print("-" * 80)
for st, row in decomp.iterrows():
    print(f"{st:<6} {row['share_first']:>7.1%} {row['share_last']:>7.1%} "
          f"{row['apr_first']:>7.2%} {row['apr_last']:>7.2%} "
          f"{row['mix_effect']*100:>+8.3f}% {row['rate_effect']*100:>+8.3f}% {row['total_effect']*100:>+8.3f}%")

print("-" * 80)
print(f"{'TOTAL':<6} {'':>8} {'':>8} {'':>8} {'':>8} "
      f"{decomp['mix_effect'].sum()*100:>+8.3f}% {decomp['rate_effect'].sum()*100:>+8.3f}% {decomp['total_effect'].sum()*100:>+8.3f}%")

Shift-Share Decomposition: 2026-01 vs 2026-07
State   Share_1  Share_T    APR_1    APR_T   Mix_Eff  Rate_Eff     Total
--------------------------------------------------------------------------------
FL       33.4%   27.7%  22.06%  20.42%   -1.210%   -0.501%   -1.710%
NV        9.6%    5.3%  27.74%  27.99%   -1.207%   +0.018%   -1.188%
AZ        8.9%    6.1%  27.85%  27.99%   -0.770%   +0.010%   -0.760%
CA        6.3%    3.8%  27.99%  27.99%   -0.702%   +0.000%   -0.702%
AL        2.4%    1.5%  27.98%  27.99%   -0.259%   +0.000%   -0.259%
TN        0.8%    0.1%  27.99%  27.99%   -0.176%   +0.000%   -0.176%
MN        0.4%    0.0%  26.41%   0.00%   -0.047%   -0.047%   -0.094%
MS        0.3%    0.1%  27.99%  27.99%   -0.073%   -0.000%   -0.073%
OH        1.1%    0.8%  25.00%  24.83%   -0.069%   -0.002%   -0.071%
IA        0.3%    0.0%  20.60%   0.00%   -0.032%   -0.032%   -0.064%
ID        0.4%    0.2%  27.99%  27.99%   -0.054%   +0.000%   -0.054%
TE        0.2%    0.0%  24.10%   0.00%   

In [11]:
# Top states driving APR decline (largest negative total_effect)
top_n = 10
top_drivers = decomp.head(top_n).copy()

print("=" * 80)
print(f"Top {top_n} States Driving APR Lower ({first_month} -> {last_month})")
print("=" * 80)
print(f"\n{'State':<6} {'Mix Effect':>12} {'Rate Effect':>12} {'Total Effect':>12} {'Diagnosis'}")
print("-" * 80)
for st, row in top_drivers.iterrows():
    if abs(row['mix_effect']) > abs(row['rate_effect']):
        diagnosis = "Volume shift"
    else:
        diagnosis = "APR decline"
    print(f"{st:<6} {row['mix_effect']*100:>+11.3f}% {row['rate_effect']*100:>+11.3f}% "
          f"{row['total_effect']*100:>+11.3f}%  {diagnosis}")

print("\n")
print("=" * 80)
print("Summary: Contribution Breakdown")
print("=" * 80)
total_mix = decomp['mix_effect'].sum()
total_rate = decomp['rate_effect'].sum()
total_all = decomp['total_effect'].sum()
if total_all != 0:
    print(f"  Mix effect (state volume shifts):  {total_mix*100:>+.3f}% ({total_mix/total_all*100:.1f}% of total change)")
    print(f"  Rate effect (within-state APR):    {total_rate*100:>+.3f}% ({total_rate/total_all*100:.1f}% of total change)")
    print(f"  Total APR change:                  {total_all*100:>+.3f}%")
else:
    print("  No APR change detected between first and last month.")

# Narrowed list of states that explain most of the decline
cumulative = decomp['total_effect'].cumsum()
total_decline = decomp['total_effect'].sum()
if total_decline < 0:
    threshold = total_decline * 0.80
    key_states = decomp[cumulative <= threshold].index.tolist()
    if len(key_states) == 0:
        key_states = [decomp.index[0]]
    remaining = total_decline - decomp.loc[key_states, 'total_effect'].sum()
    print(f"\n  States explaining >= 80% of decline: {key_states}")
    print(f"  These {len(key_states)} states account for {decomp.loc[key_states, 'total_effect'].sum()/total_decline*100:.1f}% of the total APR drop.")

Top 10 States Driving APR Lower (2026-01 -> 2026-07)

State    Mix Effect  Rate Effect Total Effect Diagnosis
--------------------------------------------------------------------------------
FL          -1.210%      -0.501%      -1.710%  Volume shift
NV          -1.207%      +0.018%      -1.188%  Volume shift
AZ          -0.770%      +0.010%      -0.760%  Volume shift
CA          -0.702%      +0.000%      -0.702%  Volume shift
AL          -0.259%      +0.000%      -0.259%  Volume shift
TN          -0.176%      +0.000%      -0.176%  Volume shift
MN          -0.047%      -0.047%      -0.094%  APR decline
MS          -0.073%      -0.000%      -0.073%  Volume shift
OH          -0.069%      -0.002%      -0.071%  Volume shift
IA          -0.032%      -0.032%      -0.064%  APR decline


Summary: Contribution Breakdown
  Mix effect (state volume shifts):  -0.601% (41.1% of total change)
  Rate effect (within-state APR):    -0.860% (58.9% of total change)
  Total APR change:                  -1

In [12]:
# Monthly weighted APR trend for the key driver states vs rest
top_states = decomp.head(5).index.tolist()

apr_df['group'] = apr_df['state'].apply(lambda s: s if s in top_states else 'All Other')

monthly_by_group = apr_df.groupby(['month_label', 'group']).agg(
    total_amt=('total_amt_financed', 'sum'),
    apr_x_amt=('apr_x_amt', 'sum'),
    loans=('loan_count', 'sum')
).reset_index()
monthly_by_group['weighted_apr'] = monthly_by_group['apr_x_amt'] / monthly_by_group['total_amt']

apr_trend = monthly_by_group.pivot_table(
    index='group', columns='month_label', values='weighted_apr'
)
count_trend = monthly_by_group.pivot_table(
    index='group', columns='month_label', values='loans', aggfunc='sum'
)

print("=" * 80)
print(f"Weighted APR Trend - Top 5 Driver States vs All Other")
print("=" * 80)
print("\nWeighted APR:")
print((apr_trend * 100).round(2).to_string())
print("\nLoan Counts:")
print(count_trend.to_string())

print("\n")
print("=" * 80)
print("Volume Share Trend - Top 5 Driver States")
print("=" * 80)
share_trend = monthly_by_group.pivot_table(
    index='group', columns='month_label', values='total_amt'
)
share_trend = share_trend.apply(lambda col: col / col.sum() * 100)
print((share_trend.round(2)).to_string())

Weighted APR Trend - Top 5 Driver States vs All Other

Weighted APR:
month_label  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
group                                                                     
AL             27.98    27.85    27.99    27.99    27.91    27.99    27.99
AZ             27.85    27.99    27.99    27.99    27.97    27.99    27.99
All Other      23.05    23.87    23.29    22.68    22.31    22.14    22.00
CA             27.99    27.99    27.99    27.24    27.28    27.99    27.99
FL             22.06    22.33    22.84    22.57    20.88    21.26    20.42
NV             27.74    27.99    27.99    27.99    27.99    27.87    27.99

Loan Counts:
month_label  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
group                                                                     
AL                18       16       28       16       27        5       12
AZ                57       67       88       69       66       61       49
All Other        